In [ ]:
from os.path import join, exists
from os import mkdir
from torch.nn import functional as F
from torchvision import transforms
import torch as th
from PIL import Image
from tqdm import tqdm

In [ ]:
zip_path = "/home/samuel/Téléchargements/vesuvius-challenge-ink-detection.zip"

In [ ]:
output_dir = "/home/samuel/PycharmProjects/VesuviusChallenge/res"

In [ ]:
if not exists(output_dir):
    mkdir(output_dir)

In [ ]:
!unzip $zip_path -d $output_dir

In [ ]:
!ls $output_dir/train

In [ ]:
DESIRED_SIZE = (256, 256)

to_tensor = transforms.ToTensor()

extracted_tensor_path = join(output_dir, "train_tensors")
if not exists(extracted_tensor_path):
    mkdir(extracted_tensor_path)

idx = 0

for img_idx in range(1, 4):
    img_folder = join(output_dir, "train", str(img_idx))
    
    label = join(img_folder, "inklabels.png")
    label_t = (
        F.unfold(
            to_tensor(Image.open(label))[None],
            DESIRED_SIZE, 1, 0, DESIRED_SIZE
        )
        .view(1, DESIRED_SIZE[0], DESIRED_SIZE[1], -1)
        .permute(3, 0, 1, 2)
        .gt(0)
        .to(th.uint8)
    )
    
    mask = join(img_folder, "mask.png")
    mask_t = (
        F.unfold(
            to_tensor(Image.open(mask))[None],
            DESIRED_SIZE, 1, 0, DESIRED_SIZE
        )
        .view(DESIRED_SIZE[0] * DESIRED_SIZE[1], -1)
        .permute(1, 0)
        .gt(0)
        .any(dim=1)
    )
    
    label_idx = idx
    for i in tqdm(range(label_t.size(0))):
        if bool(mask_t[i]):
            th.save(
                label_t[i], join(extracted_tensor_path, f"mask_{label_idx}.pt")
            )
    
            label_idx += 1
    
    slices_t = []
    
    for slice_idx in tqdm(range(1, 65)):
        slice_path = join(img_folder, "surface_volume", f"{slice_idx:02}.tif")
        
        slices_t.append(
            F.unfold(
                to_tensor(Image.open(slice_path)),
                DESIRED_SIZE, 1, 0, DESIRED_SIZE
            )
            .view(1, DESIRED_SIZE[0], DESIRED_SIZE[1], -1)
            .permute(3, 0, 1, 2)
            .to(th.int16)
        )
    
    slices_t = th.stack(slices_t, dim=4)
    
    img_idx = idx
    
    for i in tqdm(range(slices_t.size(0))):
        if bool(mask_t[i]):
            th.save(
                slices_t[i], join(extracted_tensor_path, f"img_{img_idx}.pt")
            )
            
            img_idx += 1
    
    assert label_idx == img_idx
    
    idx = img_idx